# 311 Service Requests from Louisville, KY in 2025

Import libraries and data

In [46]:
import pandas as pd

# on initial read of data file, Python informs that the columns listed below in dtype_fix are mixed. 
# so first, we assign them a string datatype. 
dtype_fix = {'source': 'str', 'description': 'str', 'zip_code': 'str', 'council_district': 'str'}
lou311 = pd.read_csv('metro_311_2025.csv', dtype=dtype_fix)


Inspect the scope and shape of the data

In [112]:
print('Rows, Columns')
print(lou311.shape)

print('====================')

print(lou311.info())

print('====================')

print('# of unique values per column:')
print(lou311.nunique())

print('====================')

print(lou311.head())

Rows, Columns
(182490, 19)
<class 'pandas.DataFrame'>
RangeIndex: 182490 entries, 0 to 182489
Data columns (total 19 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   service_request_id  182490 non-null  str    
 1   requested_datetime  182490 non-null  str    
 2   probyear            182490 non-null  int64  
 3   updated_datetime    136948 non-null  str    
 4   closed_date         122547 non-null  str    
 5   status_description  182490 non-null  str    
 6   status_notes        0 non-null       float64
 7   source              106785 non-null  str    
 8   service_name        182490 non-null  str    
 9   description         115872 non-null  str    
 10  agency_responsible  20296 non-null   str    
 11  address             129116 non-null  str    
 12  longitude           182490 non-null  float64
 13  latitude            182490 non-null  float64
 14  zip_code            127137 non-null  str    
 15  council_district  

---
## Data Cleaning

Which columns have missing data?

In [48]:
print(lou311.isnull().sum())

service_request_id         0
requested_datetime         0
probyear                   0
updated_datetime       45542
closed_date            59943
status_description         0
status_notes          182490
source                 75705
service_name               0
description            66618
agency_responsible    162194
address                53374
longitude                  0
latitude                   0
zip_code               55353
council_district       49577
ObjectId                   0
x                          0
y                          0
dtype: int64


**Initial Observations**
* status_notes seems to be an empty column. number of null values equals total number of entries.
* to verify this, we see that service_request_id has zero null values, meaning every row has a service_request_id. so:

In [56]:
print('Total # entries in file: ',lou311['service_request_id'].count())
print('Total null values in status_notes: ', lou311['status_notes'].isnull().sum())
print('Do these numbers match? ', lou311['service_request_id'].count() == lou311['status_notes'].isnull().sum())



Total # entries in file:  182490
Total null values in status_notes:  182490
Do these numbers match?  True


e.g. Drop status_notes from consideration. 
In so doing, start a new cleaned dataframe.

In [90]:
lou311_clean = lou311.drop(labels = 'status_notes', axis='columns')
lou311_clean.columns

Index(['service_request_id', 'requested_datetime', 'probyear',
       'updated_datetime', 'closed_date', 'status_description', 'source',
       'service_name', 'description', 'agency_responsible', 'address',
       'longitude', 'latitude', 'zip_code', 'council_district', 'ObjectId',
       'x', 'y'],
      dtype='str')

---
**Other Columns to Omit?**

While we're at it, the source webpage at data.louisvilleky.gov makes no mention of columns ObjectId, x, or y in the full details page for the 2025 311 data.
A glance at ObjectID reveals it to be a simple counter, and so functionally indistinguishable from the baseline index values.

In [65]:
print(lou311_clean['ObjectId'].head())

0    1
1    2
2    3
3    4
4    5
Name: ObjectId, dtype: int64


e.g. We drop ObjectId as well.

In [91]:
lou311_clean = lou311_clean.drop(labels = 'ObjectId', axis='columns')
lou311_clean.columns

Index(['service_request_id', 'requested_datetime', 'probyear',
       'updated_datetime', 'closed_date', 'status_description', 'source',
       'service_name', 'description', 'agency_responsible', 'address',
       'longitude', 'latitude', 'zip_code', 'council_district', 'x', 'y'],
      dtype='str')

Since the source page also does not mention columns x and y, what can we determine about them? The values they contain do not make it as obvious as it was with ObjectId.
From the .nunique output, we can observe that x has the same number of entries as longitude, and y has the same number of entries as latitude.
Are they, in fact, the same?

In [109]:
print('# of unique values per column:')
print(lou311.nunique())
# sort both x and longitude then compare
x_asc = lou311['x'].sort_values()
long_asc = lou311['longitude'].sort_values()
print('Column x = column longitude: ', x_asc.equals(long_asc))
# sort both y and latitude then compare
y_asc = lou311['y'].sort_values()
lat_asc = lou311['latitude'].sort_values()
print('Column y = column latitude: ', y_asc.equals(lat_asc))

# of unique values per column:
service_request_id    182488
requested_datetime       365
probyear                   1
updated_datetime         447
closed_date              447
status_description         2
status_notes               0
source                     2
service_name              55
description           103137
agency_responsible        35
address                68434
longitude              67199
latitude               67134
zip_code                  43
council_district          28
ObjectId              182490
x                      67199
y                      67134
dtype: int64
Column x = column longitude:  False
Column y = column latitude:  False


x & y are apparently *not* longitude & latitude. 
The number of unique values matching between this pair of pairs is likely more than coincidental. But since we cannot reliably identify *what* they are, they are dropped from the clean data set. We can refer back to the original data set if needed later.

In [110]:
lou311_clean = lou311_clean.drop(labels = ['x', 'y'], axis='columns')
lou311_clean.columns

Index(['service_request_id', 'requested_datetime', 'probyear',
       'updated_datetime', 'closed_date', 'status_description', 'source',
       'service_name', 'description', 'agency_responsible', 'address',
       'longitude', 'latitude', 'zip_code', 'council_district'],
      dtype='str')

Take another look at the current cleaned dataframe:

In [113]:
print(lou311_clean.info())

<class 'pandas.DataFrame'>
RangeIndex: 182490 entries, 0 to 182489
Data columns (total 15 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   service_request_id  182490 non-null  str    
 1   requested_datetime  182490 non-null  str    
 2   probyear            182490 non-null  int64  
 3   updated_datetime    136948 non-null  str    
 4   closed_date         122547 non-null  str    
 5   status_description  182490 non-null  str    
 6   source              106785 non-null  str    
 7   service_name        182490 non-null  str    
 8   description         115872 non-null  str    
 9   agency_responsible  20296 non-null   str    
 10  address             129116 non-null  str    
 11  longitude           182490 non-null  float64
 12  latitude            182490 non-null  float64
 13  zip_code            127137 non-null  str    
 14  council_district    132913 non-null  str    
dtypes: float64(2), int64(1), str(12)
memory usage

---
**Date Formats**

Several columns represent dates, including requested_datetime, probyear, updated_datetime, and closed_date. 
probyear is simply the year value. In this case, 2025 is the only value in that column, since this is 2025 annual data. We could drop it, but since we may integrate prior years or 2026 YTD later, it should stay.
If we need to examine the relationships between requested_datetime (time the request was made) and updated_datetime (when the request status was updated) or closed_date (date the request was closed), we should probably have pandas read those columns as datetime when reading in the CSV.

---
Are there any truly duplicate rows?

In [111]:
lou311_clean.duplicated().sum()

np.int64(0)

* No duplicate rows.

---
**Further investigation on ZIP codes**
* Many entries are missing zip_code, but no entires are missing latitude or longitude. 
* Can zip_code be derived from latitude/longitude combination? If so, how? 
* Could the same process to backfill zip_code be used for council_district?

In [115]:
# Take a closer look at ZIP code data. 
# How many ZIP codes are represented?
# Which ZIP code is the most frequent user of 311 services?

lou311['zip_code'] = lou311['zip_code'].astype(str)
# print(lou311['zip_code'].dtypes)
# ^^ confirms change of zip_code to string

missing_zips = lou311['zip_code'].isnull().sum()
pct_missing_zips = lou311['zip_code'].isnull().sum() / lou311['service_request_id'].value_counts().sum() * 100

print('# of entries missing ZIP codes: ',  "{:,}".format(missing_zips))
print('% of entries missing ZIP codes: ', "{:.2f}".format(pct_missing_zips), "%")

print("====================")

print(lou311['zip_code'].describe())

print("====================")



# of entries missing ZIP codes:  55,353
% of entries missing ZIP codes:  30.33 %
count     127137
unique        43
top        40211
freq        9612
Name: zip_code, dtype: object
